# Building a Q&A Retrieval-Augmented Generation (RAG) Pipeline

## 1. Setup & Configuration

In [12]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppresses TensorFlow warnings
from dotenv import load_dotenv
import Stemmer
import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from llama_index.core import Document
from llama_index.core import (SimpleDirectoryReader, VectorStoreIndex, Settings)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter
import asyncio


load_dotenv()  # Load environment variables from .env file
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    print("✅ API Key Loaded Successfully:", api_key[:5] + "..." + api_key[-5:])
else:
    print("⚠️ API Key is missing! Check your .env file.")


# ✅ Initialize OpenRouter LLM
llm = OpenRouter(api_key=api_key, model="meta-llama/llama-3.1-8b-instruct", max_tokens=512, context_window=128000)
Judge_llm = OpenRouter(api_key=api_key, model="qwen/qwen-turbo", max_tokens=512, context_window=4096)
Settings.llm = llm

# ✅ Apply nest_asyncio to fix event loop issues in Jupyter
nest_asyncio.apply()

# ✅ Set up embedding model
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = HuggingFaceEmbedding(model_name=embed_model_name)
Settings.embed_model = embed_model

✅ API Key Loaded Successfully: sk-or...01ac9


## 2. Document Loading & Preprocessing

In [13]:
# 📌 Task 1: Optimizing the Ingestion Pipeline for Medical Documents
# Load and process WebMD PDF into existing RAG pipeline
pdf_path = "./WebMD.pdf"

# Things to change
# - chunk size (small vs large)
# - chunk overlap
# - SentenceSplitter vs paragraph separated splitter
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

splitter = SentenceSplitter(
    #paragraph_separator="\n\n", # separates paragraphs by two new lines
    chunk_size=1024,
    chunk_overlap=32
)

# 📌 Task 3: Extracting Information from Images
from PIL import Image, ImageOps
import pytesseract
from pdf2image import convert_from_path

images = convert_from_path(pdf_path, first_page=1, last_page=9)

text = []

for image in images:
    text.append(pytesseract.image_to_string(image))

documents = [Document(text=t) for t in text if t.strip()]
pipeline = IngestionPipeline(transformations=[splitter, embed_model])
nodes = pipeline.run(documents=documents)
print(f"Number of nodes extracted: {len(nodes)}")

Number of nodes extracted: 9


## 3. Indexing & Retrieval

In [25]:
# ✅ Create Vector Index and Query Engine
index = VectorStoreIndex(nodes)
#query_engine = index.as_query_engine(verbose=False)

# ✅ Create base retrievers
base_retriever = index.as_retriever(similarity_top_k=1)

from llama_index.core.retrievers import AutoMergingRetriever
auto_base_retriever = index.as_retriever(similarity_top_k=3)
auto_merging_retriever = AutoMergingRetriever(auto_base_retriever, index.storage_context)

from llama_index.retrievers.bm25 import BM25Retriever
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=2, stemmer=Stemmer.Stemmer("english"), language="english")


# 📌 Task 4: Implementing a Hybrid Retrieval Approach
# Vector-based retriever (now 5, originally 1)
vector_retriever = index.as_retriever(similarity_top_k=5)

from llama_index.core.retrievers import QueryFusionRetriever
# Used Gemini for debugging the llama model
qf_retriever = QueryFusionRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    similarity_top_k=5,
    num_queries=2,  # Reduced from 4 to 2 to minimize noise
    mode="relative_score", # Better than reciprocal_rerank for medical data
    use_async=True,
    verbose=False,
    llm=llm # Explicitly pass Llama model here
)

## 4. Query Engines & Evaluation Setup

In [26]:
# ✅ Create query engines
from llama_index.core.query_engine import RetrieverQueryEngine

# 📌 Task 5b: Improving RAG with Postprocessing Nodes with LongContextReorder
from llama_index.core.postprocessor import LongContextReorder
reorder = LongContextReorder()

base_query_engine = RetrieverQueryEngine.from_args(base_retriever, verbose=False)
auto_query_engine = RetrieverQueryEngine.from_args(auto_merging_retriever, verbose=False)
bm25_query_engine = RetrieverQueryEngine.from_args(bm25_retriever, verbose=False)
qf_query_engine = RetrieverQueryEngine.from_args(qf_retriever, verbose=False)

# From Google Gemini to fix discrepancy with model change
from llama_index.core import PromptTemplate

qa_prompt_tmpl_str = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "You are a helpful assistant. Use the following context to answer the question.\n"
    "Context:\n{context_str}<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>\n\n"
    "{query_str}<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
)
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

# Apply it to your engines
base_query_engine.update_prompts({"response_synthesizer:text_qa_template": qa_prompt_tmpl})
auto_query_engine.update_prompts({"response_synthesizer:text_qa_template": qa_prompt_tmpl})
bm25_query_engine.update_prompts({"response_synthesizer:text_qa_template": qa_prompt_tmpl})
qf_query_engine.update_prompts({"response_synthesizer:text_qa_template": qa_prompt_tmpl})

# ✅ Initialize Evaluators
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator, RetrieverEvaluator

faithfulness_evaluator = FaithfulnessEvaluator(llm=Judge_llm) # using same llm as before
relevancy_evaluator = RelevancyEvaluator(llm=Judge_llm) # using same llm as before

# ✅ Define retriever evaluators
base_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=base_retriever)
auto_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=auto_merging_retriever)
bm25_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=bm25_retriever)
qf_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=qf_retriever)

## 5. Preparing Evaluation Questions & Display Functions

In [27]:
# 📌 Task 2: Defining a Comprehensive RAG Test Set
# ✅ Use Evaluation Questions
eval_questions = []


# 📌 Task 5a: Improving RAG with Postprocessing Nodes with Alternative Questions
# Paraphrase function
def paraphrase_query(query):
  qf_response = qf_query_engine.query(
      f"Paraphrase this question: {query} to create 2 alternative versions that ask the same thing to improve retrieval coverage. Put each question on a new line and only include the new questions (no introduction) so the response should be 2 lines total.")
  return qf_response.response.split("\n")

with open('my_questions.txt', 'r') as file:
    for line in file:
        stripped_line = line.strip()
        if stripped_line:
            eval_questions.append(stripped_line)

            # create alternative questions
            paraphrases = paraphrase_query(stripped_line)
            for paraphrase in paraphrases:
                cleaned_paraphrase = paraphrase.strip()
                if cleaned_paraphrase and not cleaned_paraphrase.lower().startswith("generated queries"):
                    eval_questions.append(cleaned_paraphrase)

print(f"Total questions: {len(eval_questions)}")

from llama_index.core.evaluation import generate_question_context_pairs
qa_dataset = generate_question_context_pairs(nodes=nodes, llm=llm, num_questions_per_chunk=1)


# ✅ Pretty Display Function
def displayify_df(df):
    """For pretty displaying DataFrame in a notebook."""
    display_df = df.style.set_properties(
        **{
            "inline-size": "300px",
            "overflow-wrap": "break-word",
        }
    )
    display(display_df)

# Helper to display retrieval metrics
def display_retriever_eval_results(name, eval_results):
    """Build a small DataFrame summarizing retrieval metrics across queries."""
    print(f"=== {name} ===")
    metric_dicts = [res.metric_vals_dict for res in eval_results]
    if not metric_dicts:
        print("No retriever metrics found!")
        return
    df = pd.DataFrame(metric_dicts)
    #displayify_df(df)
    print("Mean:\n", df.mean(numeric_only=True), "\n")

Total questions: 45


100%|██████████| 9/9 [01:55<00:00, 12.79s/it]


## 6. Running Evaluations & Comparing Retrievers

In [28]:
# Evaluate all retrievers on the QA dataset
# ✅ Modify Evaluation to Include Fusion Retriever
async def run_evaluation():
    eval_results = []

    print("Evaluating")

    for i, query in enumerate(eval_questions):
        print(f"Processing question {i}...")
        # Retrieve responses from all four retrievers
        base_response = base_query_engine.query(query)
        auto_response = auto_query_engine.query(query)
        bm25_response = bm25_query_engine.query(query)
        qf_response = qf_query_engine.query(query)

        base_text = base_response.response
        auto_text = auto_response.response
        bm25_text = bm25_response.response
        qf_text = qf_response.response

        base_contexts = "\n".join([node.get_content() for node in base_response.source_nodes])
        auto_contexts = "\n".join([node.get_content() for node in auto_response.source_nodes])
        bm25_contexts = "\n".join([node.get_content() for node in bm25_response.source_nodes])
        qf_contexts = "\n".join([node.get_content() for node in qf_response.source_nodes])

        # Evaluate Faithfulness & Relevancy for each retriever
        base_faithfulness = faithfulness_evaluator.evaluate_response(response=base_response)
        auto_faithfulness = faithfulness_evaluator.evaluate_response(response=auto_response)
        bm25_faithfulness = faithfulness_evaluator.evaluate_response(response=bm25_response)
        qf_faithfulness = faithfulness_evaluator.evaluate_response(response=qf_response)

        base_relevancy = relevancy_evaluator.evaluate_response(query=query, response=base_response)
        auto_relevancy = relevancy_evaluator.evaluate_response(query=query, response=auto_response)
        bm25_relevancy = relevancy_evaluator.evaluate_response(query=query, response=bm25_response)
        qf_relevancy = relevancy_evaluator.evaluate_response(query=query, response=qf_response)

        eval_results.append({
            "Query": query,
            "Base Response": base_text,
            "Auto-Merged Response": auto_text,
            "BM25 Response": bm25_text,
            "Fusion Response": qf_text,
            "Base Context": "".join(base_contexts[:100]) + "... " + "".join(base_contexts[-100:]),
            "Auto Context": "".join(auto_contexts[:100]) + "... " + "".join(auto_contexts[-100:]),
            "BM25 Context": "".join(bm25_contexts[:100]) + "... " + "".join(bm25_contexts[-100:]),
            "Fusion Context": "".join(qf_contexts[:100]) + "... " + "".join(qf_contexts[-100:]),
            "Base Faithfulness": base_faithfulness.score,
            "Auto Faithfulness": auto_faithfulness.score,
            "BM25 Faithfulness": bm25_faithfulness.score,
            "Fusion Faithfulness": qf_faithfulness.score,
            "Base Relevancy": base_relevancy.score,
            "Auto Relevancy": auto_relevancy.score,
            "BM25 Relevancy": bm25_relevancy.score,
            "Fusion Relevancy": qf_relevancy.score,
        })

    df = pd.DataFrame(eval_results)

    # Compute means for comparison
    print("\n✅ Faithfulness Comparison")
    print(f"Base Retriever: {df['Base Faithfulness'].mean():.4f}")
    print(f"Auto-Merging Retriever: {df['Auto Faithfulness'].mean():.4f}")
    print(f"BM25 Retriever: {df['BM25 Faithfulness'].mean():.4f}")
    print(f"Fusion Retriever: {df['Fusion Faithfulness'].mean():.4f}")

    print("\n✅ Relevancy Comparison")
    print(f"Base Retriever: {df['Base Relevancy'].mean():.4f}")
    print(f"Auto-Merging Retriever: {df['Auto Relevancy'].mean():.4f}")
    print(f"BM25 Retriever: {df['BM25 Relevancy'].mean():.4f}")
    print(f"Fusion Retriever: {df['Fusion Relevancy'].mean():.4f}")

    # Evaluate all retrievers on the QA dataset
    base_eval_results = await base_retriever_evaluator.aevaluate_dataset(qa_dataset)
    auto_eval_results = await auto_retriever_evaluator.aevaluate_dataset(qa_dataset)
    bm25_eval_results = await bm25_retriever_evaluator.aevaluate_dataset(qa_dataset)
    qf_eval_results = await qf_retriever_evaluator.aevaluate_dataset(qa_dataset)

    print("=== Retrieval Metrics Comparison ===")
    display_retriever_eval_results("Base Retriever", base_eval_results)
    display_retriever_eval_results("Auto-Merging Retriever", auto_eval_results)
    display_retriever_eval_results("BM25 Retriever", bm25_eval_results)
    display_retriever_eval_results("Fusion Retriever", qf_eval_results)

    # Display results table
    displayify_df(df)

    df.to_csv('results.csv', index=False)

# ✅ Execute Async Evaluation
asyncio.run(run_evaluation())

Evaluating
Processing question 0...
Processing question 1...
Processing question 2...
Processing question 3...
Processing question 4...
Processing question 5...
Processing question 6...
Processing question 7...
Processing question 8...
Processing question 9...
Processing question 10...
Processing question 11...
Processing question 12...
Processing question 13...
Processing question 14...
Processing question 15...
Processing question 16...
Processing question 17...
Processing question 18...
Processing question 19...
Processing question 20...
Processing question 21...
Processing question 22...
Processing question 23...
Processing question 24...
Processing question 25...
Processing question 26...
Processing question 27...
Processing question 28...
Processing question 29...
Processing question 30...
Processing question 31...
Processing question 32...
Processing question 33...
Processing question 34...
Processing question 35...
Processing question 36...
Processing question 37...
Processing 


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/miniconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/miniconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/miniconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/opt/miniconda3/lib/python3.12/s

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



,Query,Base Response,Auto-Merged Response,BM25 Response,Fusion Response,Base Context,Auto Context,BM25 Context,Fusion Context,Base Faithfulness,Auto Faithfulness,BM25 Faithfulness,Fusion Faithfulness,Base Relevancy,Auto Relevancy,BM25 Relevancy,Fusion Relevancy
0,Can you list all of the causes or aggravating factors of migraines from this article?,"Unfortunately, the article does not explicitly list the causes or aggravating factors of migraines. However, I can infer some potential causes and aggravating factors based on general knowledge and the context of the article. Here are some possible causes and aggravating factors of migraines that may be related to the article: 1. **Lifestyle changes**: The article mentions ""Prioritize lifestyle changes"" as a section title, suggesting that lifestyle factors may play a role in managing migraines. 2. **Stress**: While not explicitly mentioned, stress is a common trigger for migraines. 3. **Hormonal changes**: Although not mentioned in the article, hormonal fluctuations are a known trigger for migraines in some people. 4. **Sleep patterns**: Irregular sleep patterns or lack of sleep can trigger migraines. 5. **Diet**: Certain foods or food additives can trigger migraines in some people. 6. **Environmental factors**: Sensitivity to light, sound, or other environmental stimuli can trigger migraines. 7. **Genetics**: Migraines can be inherited, suggesting a genetic component. Please note that these are general potential causes and aggravating factors, and the article does not provide specific information on these topics. If you're looking for more information on migraine causes and triggers, I recommend consulting a healthcare professional or a reliable medical resource.","Based on the article, the following are the causes or aggravating factors of migraines mentioned: 1. Hormonal changes (e.g. during pregnancy) 2. Smells 3. Lights 4. Mercury and lead levels 5. Mold exposure 6. Comorbidities: * Mood disorders (depression and anxiety) * Sleep disorders (sleep apnea, insomnia, restless legs syndrome) * Cardiovascular disorders (heart attack, stroke, and death from cardiovascular disease) * Inflammatory conditions (arthritis) * Neurological disorders (epilepsy, multiple sclerosis, stroke) * Fibromyalgia * Gastrointestinal issues (nausea, diarrhea, vomiting, IBD, IBS) Note that these are not necessarily the only causes or aggravating factors of migraines, but rather a list of the specific ones mentioned in the article.","According to the article, the following are the aggravating factors or comorbidities associated with migraines: 1. **Mood disorders**: * Depression * Anxiety 2. **Sleep disorders**: * Sleep apnea * Insomnia * Restless legs syndrome * Poor sleep quality and duration 3. **Cardiovascular disorders**: * Heart attack * Stroke * Death from cardiovascular disease (for people with chronic migraine with aura) 4. **Inflammatory conditions**: * Arthritis 5. **Neurological disorders**: * Epilepsy * Multiple sclerosis * Stroke 6. **Chronic pain**: * Fibromyalgia 7. **Gastrointestinal issues**: * Nausea * Diarrhea * Vomiting * Inflammatory bowel disease (IBD) * Irritable bowel syndrome (IBS) Additionally, the article mentions that physical activity can also trigger migraines, but it's not listed as a comorbidity.","Based on the article, the following are the causes or aggravating factors of migraines mentioned: **Comorbidities:** 1. Mood disorders (depression and anxiety) 2. Sleep disorders (sleep apnea, insomnia, restless legs syndrome) 3. Cardiovascular disorders (heart attack, stroke, death from cardiovascular disease) 4. Inflammatory conditions (arthritis) 5. Neurological disorders (epilepsy, multiple sclerosis, stroke) 6. Chronic pain (fibromyalgia) 7. Gastrointestinal issues (nausea, diarrhea, vomiting, inflammatory bowel disease (IBD), irritable bowel syndrome (IBS)) **Other factors:** 1. Hormonal changes (during pregnancy) 2. Smells 3. Lights 4. Physical activity 5. Stress 6. Poor